
# Caso de clase CAPM — Parte 3
## CAPM vs. Fama–French de tres factores

Esta tercera parte extiende el análisis del CAPM para estudiar una de sus principales debilidades empíricas:

> **¿Es el riesgo de mercado, medido por beta, suficiente para explicar los retornos?**

El CAPM propone:

$$
R_{i,t}-R_{f,t}
=
\alpha_i
+
\beta_{MKT,i}(R_{m,t}-R_{f,t})
+
\varepsilon_{i,t}
$$

El modelo Fama–French de tres factores agrega dos fuentes adicionales de variación:

$$
R_{i,t}-R_{f,t}
=
\alpha_i
+
\beta_{MKT,i}MKT_t
+
\beta_{SMB,i}SMB_t
+
\beta_{HML,i}HML_t
+
\varepsilon_{i,t}
$$

donde:

$$
MKT_t = R_{m,t}-R_{f,t}
$$

$$
SMB_t = R_{\text{Small},t}-R_{\text{Big},t}
$$

$$
HML_t = R_{\text{High B/M},t}-R_{\text{Low B/M},t}
$$

El objetivo de la clase es comparar CAPM y Fama–French utilizando las mismas acciones, ETF y fondos.


In [ ]:
# Si hace falta instalar:
# !pip install yfinance pandas numpy matplotlib statsmodels pandas_datareader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

from pandas_datareader import data as web

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Activos del caso

In [ ]:
start_date = "2018-01-01"
end_date   = "2025-12-31"

stocks = {
    "Tesla": "TSLA",
    "Nvidia": "NVDA",
    "Berkshire_B": "BRK-B",
    "Booking": "BKNG",
    "Ford": "F",
    "Snap": "SNAP",
    "CocaCola": "KO",
    "JPMorgan": "JPM"
}

etfs = {
    "SP500": "SPY",
    "Nasdaq100": "QQQ",
    "SmallCap": "IWM",
    "Technology": "XLK",
    "Healthcare": "XLV",
    "Energy": "XLE"
}

funds = {
    "Vanguard500Fund": "VFIAX",
    "VanguardTotalMarket": "VTSAX",
    "FidelityContrafund": "FCNTX"
}

all_assets = {**stocks, **etfs, **funds}
ticker_to_name = {v: k for k, v in all_assets.items()}

## 2. Descargar precios mensuales

In [ ]:
def download_close(ticker, start, end):
    data = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    if data.empty:
        return None

    close = data["Close"]

    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    close.name = ticker
    return close

series = []

for ticker in all_assets.values():
    s = download_close(ticker, start_date, end_date)
    if s is not None:
        series.append(s)

prices = pd.concat(series, axis=1)
prices = prices.rename(columns=ticker_to_name)

monthly_prices = prices.resample("ME").last()
returns = monthly_prices.pct_change(fill_method=None)

print("Activos disponibles:")
print(list(returns.columns))

returns.tail()

## 3. Descargar factores Fama–French

Se utiliza la base mensual de Kenneth French:

- `Mkt-RF`: prima de mercado;
- `SMB`: factor tamaño;
- `HML`: factor value;
- `RF`: tasa libre de riesgo.

Los factores vienen expresados en porcentaje, por lo que se dividen entre 100.


In [ ]:
ff = web.DataReader(
    "F-F_Research_Data_Factors",
    "famafrench",
    start=start_date,
    end=end_date
)[0]

ff.index = ff.index.to_timestamp("M")
ff = ff / 100

ff.head()

## 4. Unir activos y factores

In [ ]:
# Ajustar el índice mensual para que coincida
returns_ff = returns.copy()
returns_ff.index = returns_ff.index.to_period("M").to_timestamp("M")

data = returns_ff.join(ff, how="inner")

print("Número de meses disponibles:", len(data))
print("Desde:", data.index.min())
print("Hasta:", data.index.max())

data.tail()

## 5. CAPM con los factores de Fama–French

Para que la comparación sea justa, el CAPM se estima usando exactamente la misma prima de mercado del dataset de Fama–French:

$$
R_{i,t}-R_{f,t}
=
\alpha_i
+
\beta_{MKT,i}(Mkt-RF)_t
+
\varepsilon_{i,t}
$$


In [ ]:
def estimate_capm_ffdata(asset, data):

    df = data[[asset, "Mkt-RF", "RF"]].dropna()

    y = df[asset] - df["RF"]
    X = sm.add_constant(df[["Mkt-RF"]])

    model = sm.OLS(y, X).fit()

    return model

## 6. Modelo Fama–French de tres factores

La especificación es:

$$
R_{i,t}-R_{f,t}
=
\alpha_i
+
\beta_{MKT,i}(Mkt-RF)_t
+
\beta_{SMB,i}SMB_t
+
\beta_{HML,i}HML_t
+
\varepsilon_{i,t}
$$


In [ ]:
def estimate_ff3(asset, data):

    df = data[[asset, "Mkt-RF", "SMB", "HML", "RF"]].dropna()

    y = df[asset] - df["RF"]

    X = sm.add_constant(
        df[["Mkt-RF", "SMB", "HML"]]
    )

    model = sm.OLS(y, X).fit()

    return model

## 7. Comparación para todos los activos

In [ ]:
rows = []
capm_models = {}
ff3_models = {}

asset_types = {}

for a in stocks:
    asset_types[a] = "Stock"

for a in etfs:
    if a != "SP500":
        asset_types[a] = "ETF"

for a in funds:
    asset_types[a] = "Fund"

for asset, asset_type in asset_types.items():

    if asset not in data.columns:
        continue

    capm = estimate_capm_ffdata(asset, data)
    ff3_model = estimate_ff3(asset, data)

    capm_models[asset] = capm
    ff3_models[asset] = ff3_model

    rows.append({
        "Asset": asset,
        "Type": asset_type,

        "CAPM_Alpha": capm.params["const"],
        "CAPM_Beta": capm.params["Mkt-RF"],
        "CAPM_R2": capm.rsquared,
        "CAPM_Adj_R2": capm.rsquared_adj,
        "CAPM_Alpha_p": capm.pvalues["const"],
        "CAPM_AIC": capm.aic,
        "CAPM_BIC": capm.bic,

        "FF3_Alpha": ff3_model.params["const"],
        "FF3_Beta_MKT": ff3_model.params["Mkt-RF"],
        "FF3_Beta_SMB": ff3_model.params["SMB"],
        "FF3_Beta_HML": ff3_model.params["HML"],
        "FF3_R2": ff3_model.rsquared,
        "FF3_Adj_R2": ff3_model.rsquared_adj,
        "FF3_Alpha_p": ff3_model.pvalues["const"],
        "FF3_AIC": ff3_model.aic,
        "FF3_BIC": ff3_model.bic
    })

comparison = pd.DataFrame(rows).set_index("Asset")

comparison.sort_values(["Type", "CAPM_Beta"], ascending=[True, False])

## 8. ¿Cuánto mejora el ajuste con Fama–French?

In [ ]:
comparison["Delta_R2"] = (
    comparison["FF3_R2"] - comparison["CAPM_R2"]
)

comparison["Delta_Adj_R2"] = (
    comparison["FF3_Adj_R2"] - comparison["CAPM_Adj_R2"]
)

comparison[
    [
        "Type",
        "CAPM_R2",
        "FF3_R2",
        "Delta_R2",
        "CAPM_Adj_R2",
        "FF3_Adj_R2",
        "Delta_Adj_R2"
    ]
].sort_values("Delta_R2", ascending=False)

Si:

$$
R^2_{FF3} > R^2_{CAPM}
$$

los factores tamaño y value explican variación adicional de los retornos.

Sin embargo, un mayor $R^2$ no significa automáticamente que el modelo sea una mejor teoría económica. Esto es precisamente parte de la discusión histórica entre CAPM y modelos multifactoriales.


In [ ]:
plot_r2 = comparison.sort_values("Delta_R2", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))

y = np.arange(len(plot_r2))

ax.scatter(plot_r2["CAPM_R2"], y, label="CAPM")
ax.scatter(plot_r2["FF3_R2"], y, label="Fama-French 3")

ax.set_yticks(y)
ax.set_yticklabels(plot_r2.index)

ax.set_xlabel("R-cuadrado")
ax.set_title("Capacidad explicativa: CAPM vs. Fama–French")
ax.legend()

plt.tight_layout()
plt.show()

## 9. ¿Qué pasa con el alpha?

In [ ]:
comparison["CAPM_Alpha_Annual"] = (
    (1 + comparison["CAPM_Alpha"])**12 - 1
)

comparison["FF3_Alpha_Annual"] = (
    (1 + comparison["FF3_Alpha"])**12 - 1
)

comparison[
    [
        "Type",
        "CAPM_Alpha_Annual",
        "CAPM_Alpha_p",
        "FF3_Alpha_Annual",
        "FF3_Alpha_p"
    ]
].sort_values("CAPM_Alpha_Annual", ascending=False)

Una pregunta central es si el alpha disminuye después de incorporar SMB y HML.

Si:

$$
|\alpha_{FF3}|
<
|\alpha_{CAPM}|
$$

podemos interpretar que una parte de lo que parecía ser rendimiento "anormal" bajo CAPM estaba asociada a exposiciones sistemáticas adicionales.


In [ ]:
alpha_plot = comparison[
    ["CAPM_Alpha_Annual", "FF3_Alpha_Annual"]
].sort_values("CAPM_Alpha_Annual")

fig, ax = plt.subplots(figsize=(10, 7))

y = np.arange(len(alpha_plot))

ax.scatter(alpha_plot["CAPM_Alpha_Annual"], y, label="CAPM")
ax.scatter(alpha_plot["FF3_Alpha_Annual"], y, label="Fama-French 3")

ax.axvline(0, linewidth=1)

ax.set_yticks(y)
ax.set_yticklabels(alpha_plot.index)

ax.set_xlabel("Alpha anualizado")
ax.set_title("Alpha CAPM vs. alpha Fama–French")
ax.legend()

plt.tight_layout()
plt.show()

## 10. Interpretar SMB: exposición a tamaño

In [ ]:
smb_table = comparison[
    ["Type", "FF3_Beta_SMB"]
].sort_values("FF3_Beta_SMB", ascending=False)

smb_table

La carga sobre SMB puede interpretarse como exposición relativa al factor tamaño:

$$
\beta_{SMB}>0
$$

indica comportamiento más parecido a empresas pequeñas.

$$
\beta_{SMB}<0
$$

indica comportamiento más parecido a empresas grandes.

Esto debe interpretarse como una **exposición estadística al factor**, no como una clasificación mecánica del activo.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

smb_table["FF3_Beta_SMB"].sort_values().plot(kind="barh", ax=ax)

ax.axvline(0, linewidth=1)
ax.set_xlabel("Carga SMB")
ax.set_title("Exposición al factor tamaño")

plt.tight_layout()
plt.show()

## 11. Interpretar HML: exposición a value vs. growth

In [ ]:
hml_table = comparison[
    ["Type", "FF3_Beta_HML"]
].sort_values("FF3_Beta_HML", ascending=False)

hml_table

La carga sobre HML aproxima exposición al estilo value/growth:

$$
\beta_{HML}>0
$$

sugiere comportamiento más parecido a activos value.

$$
\beta_{HML}<0
$$

sugiere comportamiento más parecido a activos growth.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

hml_table["FF3_Beta_HML"].sort_values().plot(kind="barh", ax=ax)

ax.axvline(0, linewidth=1)
ax.set_xlabel("Carga HML")
ax.set_title("Exposición al factor value")

plt.tight_layout()
plt.show()

## 12. Mapa de estilos SMB–HML

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    comparison["FF3_Beta_SMB"],
    comparison["FF3_Beta_HML"]
)

for asset in comparison.index:
    ax.annotate(
        asset,
        (
            comparison.loc[asset, "FF3_Beta_SMB"],
            comparison.loc[asset, "FF3_Beta_HML"]
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)

ax.set_xlabel("Beta SMB")
ax.set_ylabel("Beta HML")
ax.set_title("Mapa de exposiciones Fama–French")

plt.tight_layout()
plt.show()

Este gráfico es especialmente útil para comparar:

- Nvidia y Tesla;
- Coca-Cola;
- JPMorgan;
- ETF tecnológicos;
- ETF small-cap;
- fondos de inversión.

Dos activos con un beta de mercado similar pueden tener exposiciones SMB y HML muy diferentes.


## 13. Comparar acciones, ETF y fondos

In [ ]:
type_summary = comparison.groupby("Type").agg(
    Mean_CAPM_R2=("CAPM_R2", "mean"),
    Mean_FF3_R2=("FF3_R2", "mean"),
    Mean_Delta_R2=("Delta_R2", "mean"),
    Mean_MKT=("FF3_Beta_MKT", "mean"),
    Mean_SMB=("FF3_Beta_SMB", "mean"),
    Mean_HML=("FF3_Beta_HML", "mean")
)

type_summary

Aquí aparece una discusión importante:

$$
\text{acción individual}
\neq
\text{ETF}
\neq
\text{fondo}
$$

aunque todos pueden analizarse con los mismos factores.

Los ETF sectoriales y de estilo suelen mostrar exposiciones factoriales más claras, mientras que una acción individual puede contener mucho más componente idiosincrático.


## 14. Ejemplo detallado: Nvidia

In [ ]:
example_asset = "Nvidia"

if example_asset in capm_models:

    print("CAPM")
    print(capm_models[example_asset].summary())

    print("\n" + "="*80 + "\n")

    print("FAMA-FRENCH 3 FACTORES")
    print(ff3_models[example_asset].summary())

Para Nvidia podemos comparar directamente:

### CAPM

$$
R_{NVDA,t}-R_{f,t}
=
\alpha
+
\beta_M(Mkt-RF)_t
+
\varepsilon_t
$$

### Fama–French

$$
R_{NVDA,t}-R_{f,t}
=
\alpha
+
\beta_M(Mkt-RF)_t
+
\beta_S SMB_t
+
\beta_H HML_t
+
\varepsilon_t
$$

La pregunta no es únicamente cuál tiene mayor $R^2$, sino:

> ¿Qué nueva interpretación económica obtenemos al introducir SMB y HML?


## 15. Comparación mediante AIC y BIC

In [ ]:
comparison[
    [
        "Type",
        "CAPM_AIC",
        "FF3_AIC",
        "CAPM_BIC",
        "FF3_BIC"
    ]
].sort_values("Type")

AIC y BIC penalizan la incorporación de parámetros adicionales.

Por tanto, permiten discutir que un modelo más complejo no debe evaluarse únicamente porque tenga:

$$
R^2
$$

más alto.

Una comparación más completa considera:

$$
\text{ajuste}
+
\text{parsimonia}
+
\text{interpretación económica}
$$


## 16. Tabla final CAPM vs. Fama–French

In [ ]:
final_table = comparison[
    [
        "Type",
        "CAPM_Beta",
        "CAPM_R2",
        "CAPM_Alpha_Annual",
        "FF3_Beta_MKT",
        "FF3_Beta_SMB",
        "FF3_Beta_HML",
        "FF3_R2",
        "FF3_Alpha_Annual",
        "Delta_R2"
    ]
].copy()

final_table.sort_values(["Type", "Delta_R2"], ascending=[True, False])

# 17. Interpretación final

La secuencia del caso ahora es:

$$
\text{CAPM}
\rightarrow
\beta_M
$$

$$
\downarrow
$$

$$
\text{¿beta explica todos los retornos?}
$$

$$
\downarrow
$$

$$
\text{Fama-French}
=
MKT+SMB+HML
$$

El estudiante puede observar tres posibles resultados:

### Caso 1

$$
R^2_{FF3}
\approx
R^2_{CAPM}
$$

Los factores adicionales aportan poco.

### Caso 2

$$
R^2_{FF3}
>
R^2_{CAPM}
$$

SMB y HML explican variación adicional.

### Caso 3

$$
\alpha_{CAPM}\neq0
\quad\text{pero}\quad
\alpha_{FF3}\approx0
$$

Parte del aparente retorno anormal del CAPM puede interpretarse como exposición a otros factores sistemáticos.

---

## Preguntas para discusión

1. ¿Por qué Nvidia y Coca-Cola pueden tener exposiciones HML tan distintas?
2. ¿Un ETF small-cap debería mostrar $\beta_{SMB}>0$?
3. ¿Un ETF tecnológico debería comportarse como growth y mostrar $\beta_{HML}<0$?
4. ¿Por qué algunos fondos pueden tener beta de mercado cercano a uno pero exposiciones SMB/HML diferentes?
5. ¿Que Fama–French ajuste mejor significa que CAPM es inútil?
6. ¿Los factores SMB y HML representan verdaderos riesgos económicos o simplemente regularidades estadísticas?
7. ¿Cuándo preferiríamos la parsimonia del CAPM frente a un modelo multifactorial?

La siguiente extensión natural sería:

$$
\text{CAPM}
\rightarrow
\text{Fama-French}
\rightarrow
\text{Fama-MacBeth}
\rightarrow
\text{crítica de Roll}
$$
